### 13. I föregående kapitel gjordes en EDA på datasetet “hr_employee_data.xlsx”. Gör nu ett komplett ML-flöde där den beroende variabeln, 𝑦, är left. Om den variabeln är 1 så betyder det att den anställde har lämnat företaget och om den är 0 så betyder det att den anställde inte har lämnat företaget, det vill säga jobbar kvar.

In [45]:
print("hello")

hello


In [46]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

### Läs in datan

In [47]:
# läsa in data
df = pd.read_excel('hr_employee_data.xlsx')
df

,Emp_Id,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,IND02438,0.38,0.53,2,157,3,0,1,0,sales,low
1,IND28133,0.80,0.86,5,262,6,0,1,0,sales,medium
2,IND07164,0.11,0.88,7,272,4,0,1,0,sales,medium
3,IND30478,0.72,0.87,5,223,5,0,1,0,sales,low
4,IND24003,0.37,0.52,2,159,3,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...,...
14994,IND40221,0.40,0.57,2,151,3,0,1,0,support,low
14995,IND24196,0.37,0.48,2,160,3,0,1,0,support,low
14996,IND33544,0.37,0.53,2,143,3,0,1,0,support,low
14997,IND40533,0.11,0.96,6,280,4,0,1,0,support,low


### Snabb koll

In [48]:
print(df.info())
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Emp_Id                 14999 non-null  str    
 1   satisfaction_level     14999 non-null  float64
 2   last_evaluation        14999 non-null  float64
 3   number_project         14999 non-null  int64  
 4   average_montly_hours   14999 non-null  int64  
 5   time_spend_company     14999 non-null  int64  
 6   Work_accident          14999 non-null  int64  
 7   left                   14999 non-null  int64  
 8   promotion_last_5years  14999 non-null  int64  
 9   Department             14999 non-null  str    
 10  salary                 14999 non-null  str    
dtypes: float64(2), int64(6), str(3)
memory usage: 1.5 MB
None
Emp_Id                   0
satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
W

In [49]:
print(df.duplicated().sum)

<bound method Series.sum of 0        False
1        False
2        False
3        False
4        False
         ...  
14994    False
14995    False
14996    False
14997    False
14998    False
Length: 14999, dtype: bool>


In [50]:
# Dela upp i x och y
x = df.drop(columns=["left", "Emp_Id"])
y = df["left"]


In [51]:
y

0        1
1        1
2        1
3        1
4        1
        ..
14994    1
14995    1
14996    1
14997    1
14998    1
Name: left, Length: 14999, dtype: int64

### One‑Hot Encoding för kategoriska variabler. Datasetet har två kategoriska features:

* Department
* salary

In [52]:
# Hantera kategoriska variabler
x = pd.get_dummies(x, drop_first=True)

In [53]:
# Train test 
x_train_val, x_test, y_train_val, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train_val, y_train_val, test_size=0.10, random_state=40)

In [54]:
# Skala numeriska värden för logistisk regression utan att läcka information från validerings- eller testdata
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

In [55]:
# Logistisk regression
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(x_train_scaled, y_train)
y_val_pred_logistic = logistic_model.predict(x_val_scaled)

print("Logistisk regression - validering")
print(classification_report(y_val, y_val_pred_logistic))
print("Konfusionsmatris:")
print(confusion_matrix(y_val, y_val_pred_logistic))

Logistisk regression - validering
              precision    recall  f1-score   support

           0       0.83      0.93      0.87       925
           1       0.59      0.35      0.44       275

    accuracy                           0.79      1200
   macro avg       0.71      0.64      0.66      1200
weighted avg       0.77      0.79      0.77      1200

Konfusionsmatris:
[[857  68]
 [179  96]]


In [56]:
# Random forest som jämförelsemodell
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
random_forest_model.fit(x_train, y_train)
y_val_pred_forest = random_forest_model.predict(x_val)

print("Random forest - validering")
print(classification_report(y_val, y_val_pred_forest))
print("Konfusionsmatris:")
print(confusion_matrix(y_val, y_val_pred_forest))

Random forest - validering
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       925
           1       0.98      0.97      0.97       275

    accuracy                           0.99      1200
   macro avg       0.99      0.98      0.98      1200
weighted avg       0.99      0.99      0.99      1200

Konfusionsmatris:
[[920   5]
 [  9 266]]


In [57]:
# Random forest valdes utifrån resultatet på valideringsdatan.
# Träna därför om den valda modellen på all träningsdata:
# train + validation = x_train_val
best_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
best_model.fit(x_train_val, y_train_val)

# Testdata används först nu, för den slutliga utvärderingen
y_test_pred = best_model.predict(x_test)

print("Random forest - slutlig utvärdering på testdata")
print(classification_report(y_test, y_test_pred))
print("Konfusionsmatris:")
print(confusion_matrix(y_test, y_test_pred))

Random forest - slutlig utvärdering på testdata
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2286
           1       0.99      0.97      0.98       714

    accuracy                           0.99      3000
   macro avg       0.99      0.98      0.99      3000
weighted avg       0.99      0.99      0.99      3000

Konfusionsmatris:
[[2280    6]
 [  24  690]]
